In [ ]:
import cutlass
import cutlass.cute as cute


@cute.kernel
def hello_kernel():
    tidx, _, _ = cute.arch.thread_idx()
    if tidx == 0:
        cute.printf("Hello from GPU")


@cute.jit
def hello_world():
    cutlass.cuda.initialize_cuda_context()
    hello_kernel().launch(grid=(1, 1, 1), block=(32, 1, 1))


compiled = cute.compile(hello_world)
compiled()

In [ ]:
@cute.jit
def print_demo(a: cutlass.Int32, b: cutlass.Constexpr[int]):
    print("static a:", a)  # => ? (dynamic)
    print("static b:", b)  # => 2
    cute.printf("dynamic a: {}", a)
    cute.printf("dynamic b: {}", b)
    layout = cute.make_layout((a, b))
    print("static layout:", layout)  # (?,2):(1,?)
    cute.printf("dynamic layout: {}", layout)  # (8,2):(1,8)


print_demo(cutlass.Int32(8), 2)

static a: ?
static b: 2
static layout: (?,2):(1,?)
dynamic a: 8
dynamic b: 2
dynamic layout: (8,2):(1,8)


Hello from GPU


Numeric types in JIT code should be explicit and can be converted at runtime

In [ ]:
@cute.jit
def dtypes():
    a = cutlass.Int32(42)
    b = a.to(cutlass.Float32)
    c = b + 0.5
    d = c.to(cutlass.Int32)
    cute.printf("a={}, b={}, c={}, d={}", a, b, c, d)


dtypes()

a=42, b=42.000000, c=42.500000, d=42


### Tensores and layouts

In [26]:
import torch
from cutlass.cute.runtime import from_dlpack


@cute.jit
def tensor_demo(t: cute.Tensor):
    cute.printf("t[0,0] = {}", t[0, 0])
    sub = t[(None, 0)]  # First row view
    frag = cute.make_rmem_tensor(sub.layout, sub.element_type)
    frag.store(sub.load())
    cute.print_tensor(frag)


arr = torch.arange(0, 12, dtype=torch.float32).reshape(3, 4)
tensor_demo(from_dlpack(arr))

t[0,0] = 0.000000
tensor(raw_ptr(0x00007ffc302325c0: f32, rmem, align<32>) o (3):(4), data=
       [ 0.000000, ],
       [ 4.000000, ],
       [ 8.000000, ])


In [27]:
@cute.jit
def layout_stride_demo(M: cutlass.Int32, N: cutlass.Int32):
    row_major = cute.make_layout((M, N), stride=(N, cutlass.Int32(1)))
    col_major = cute.make_layout((M, N), stride=(cutlass.Int32(1), M))
    print("static row-major:", row_major)
    print("static col-major:", col_major)
    cute.printf("dynamic row-major: {}", row_major)
    cute.printf("dynamic col-major: {}", col_major)

layout_stride_demo(cutlass.Int32(4), cutlass.Int32(3))

static row-major: (?,?):(?,?)
static col-major: (?,?):(?,?)
dynamic row-major: (4,3):(3,1)
dynamic col-major: (4,3):(1,4)


In [30]:
torch.arange(12, dtype=torch.float32).reshape(4, 3)

tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [ 6.,  7.,  8.],
        [ 9., 10., 11.]])

In [34]:
@cute.jit
def slicing_examples(t: cute.Tensor):
    # Scalar access
    cute.printf("t[1,2] = {}", t[1, 2])

    # Entire second col (shape: (M,)) using (None, col_index)
    col = t[(None, 1)]
    col_frag = cute.make_rmem_tensor(col.layout, col.element_type)
    col_frag.store(col.load())
    print("Second col:")
    cute.print_tensor(col_frag)

    # Entire third row (shape: (N,)) using (row_index, None)
    row = t[(2, None)]
    row_frag = cute.make_rmem_tensor(row.layout, row.element_type)
    row_frag.store(row.load())
    print("Third row:")
    cute.print_tensor(row_frag)

    # Printing the third row directly (*t[2] == *t[2, 0])
    cute.printf(
        "t[2] = {} (equivalent to t[{}])",
        t[2],
        cute.make_identity_tensor(t.layout.shape)[2]
    )

    cute.printf(t.layout)

# 4x3 example tensor
arr = torch.arange(12, dtype=torch.float32).reshape(4, 3)
slicing_examples(from_dlpack(arr))

Second col:
Third row:
t[1,2] = 5.000000
tensor(raw_ptr(0x00007ffc302325c0: f32, rmem, align<32>) o (4):(3), data=
       [ 1.000000, ],
       [ 4.000000, ],
       [ 7.000000, ],
       [ 10.000000, ])
tensor(raw_ptr(0x00007ffc302325a0: f32, rmem, align<32>) o (3):(1), data=
       [ 6.000000, ],
       [ 7.000000, ],
       [ 8.000000, ])
t[2] = 6.000000 (equivalent to t[(2,0)])
(4,3):(3,1)


## Register tensors and reductions

`TensorSSA` is a register-level value enabling vectorized elementwise ops

In [35]:
import numpy as np


@cute.jit
def ssa_add(dst: cute.Tensor, x: cute.Tensor, y: cute.Tensor):
    xv = x.load()
    yv = y.load()
    dst.store(xv + yv)
    cute.print_tensor(dst)

X = np.ones((2, 3), dtype=np.float32)
Y = np.full((2, 3), 2.0, dtype=np.float32)
Z = np.zeros((2, 3), dtype=np.float32)
ssa_add(from_dlpack(Z), from_dlpack(X), from_dlpack(Y))

tensor(raw_ptr(0x00005647b9759120: f32, generic, align<4>) o (2,3):(3,1), data=
       [[ 3.000000,  3.000000,  3.000000, ],
        [ 3.000000,  3.000000,  3.000000, ]])


Reduce arguments

- `op`: A cute.ReductionOp enum specifying the operation (ADD, MUL, MAX, MIN, …)
- `init`: Initial accumulator value (also sets accumulator dtype)
- `reduction_profile`: Axes to reduce — 0 for all axes; or a tuple with 1 to reduce / None to keep

In [39]:
@cute.jit
def ssa_reduce(a: cute.Tensor):
    v = a.load()
    # Sum of all elements
    total = v.reduce(cute.ReductionOp.ADD, 0.0, reduction_profile=0)
    cute.printf("total sum = {}", total)

    # Row-wise sum -> shape (rows,)
    row_sum = v.reduce(cute.ReductionOp.ADD, 0.0, reduction_profile=(None, 1))  # reduce the second dim - reduce columns
    row_frag = cute.make_rmem_tensor(row_sum.shape, cutlass.Float32)
    row_frag.store(row_sum)
    cute.printf("Row-wise sum:")
    cute.print_tensor(row_frag)

    # Column-wise sum -> shape (cols,)
    col_sum = v.reduce(cute.ReductionOp.ADD, 0.0, reduction_profile=(1, None))  # reduce the first dim - sum across rows for each columns
    col_frag = cute.make_rmem_tensor(col_sum.shape, cutlass.Float32)
    col_frag.store(col_sum)
    cute.printf("Column-wise sum:")
    cute.print_tensor(col_frag)

A = np.array([[1, 2, 3], [4, 5, 6]], dtype=np.float32)
ssa_reduce(from_dlpack(A))


total sum = 21.000000
Row-wise sum:
tensor(raw_ptr(0x00007ffc30232520: f32, rmem, align<32>) o (2):(1), data=
       [ 6.000000, ],
       [ 15.000000, ])
Column-wise sum:
tensor(raw_ptr(0x00007ffc30232540: f32, rmem, align<32>) o (3):(1), data=
       [ 5.000000, ],
       [ 7.000000, ],
       [ 9.000000, ])


## Vectorized add with zipped_divide

Naively mapping one thread per element is simple but not fast. We partition tensors into per-thread tiles with `cute.zipped_divide`, slice a specific tile `(None, (mi, ni))`, load vectors, compute, and store.

Tile semantics (`zipped_divide`)

- `cute.zipped_divide(tensor, (1, 4))` produces a 2-mode tiled tensor: mode-0 is the per-thread tile `(1,4)` and mode-1 indexes tiles across the original tensor.
- Printed as `(shape):(stride)` by mode, e.g. the tiled tensor may show: `((1,4),(2048,512)):((0,1),(2048,4))`
- Slicing with `(None, (mi, ni))` selects a single per-thread tile, yielding a (1,4) vector on the right-hand side.

In [49]:
@cute.kernel
def vadd_kernel(gA: cute.Tensor, gB: cute.Tensor, gC: cute.Tensor):
    tidx, _, _ = cute.arch.thread_idx()
    bidx, _, _ = cute.arch.block_idx()
    bdim, _, _ = cute.arch.block_dim()
    idx = bidx * bdim + tidx
    m, n = gA.shape[1]          # thread-domain
    mi = idx // n
    ni = idx % n
    gC[(None, (mi, ni))] = gA[(None, (mi, ni))].load() + gB[(None, (mi, ni))].load()

@cute.jit
def vadd(A: cute.Tensor, B: cute.Tensor, C: cute.Tensor):
    gA = cute.zipped_divide(A, (1, 4))
    gB = cute.zipped_divide(B, (1, 4))
    gC = cute.zipped_divide(C, (1, 4))
    # cute.printf(gC.layout)
    threads = 256
    vadd_kernel(gA, gB, gC).launch(
        grid=(cute.size(gC, mode=[1]) // threads, 1, 1),
        block=(threads, 1, 1),
    )

M, N = 1024, 1024
a = torch.randn(M, N, device="cuda", dtype=torch.float16)
b = torch.randn(M, N, device="cuda", dtype=torch.float16)
c2 = torch.zeros(M, N, device="cuda", dtype=torch.float16)
vadd_compiled = cute.compile(vadd, from_dlpack(a), from_dlpack(b), from_dlpack(c))
vadd_compiled(from_dlpack(a), from_dlpack(b), from_dlpack(c))
